In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

In [43]:
path_proj = "/content/drive/MyDrive/llm_from_scratch/src"
path_proj_datasets = "/content/drive/MyDrive/llm_from_scratch/datasets"

In [44]:
os.chdir(path_proj)
print(os.getcwd())
os.chdir(path_proj_datasets)
print(os.getcwd())

/content/drive/MyDrive/llm_from_scratch/src
/content/drive/MyDrive/llm_from_scratch/datasets


In [ ]:
import urllib.request
url = (
"https://raw.githubusercontent.com/rasbt/"
"LLMs-from-scratch/main/ch05/"
"01_main-chapter-code/gpt_download.py"
)

download_path = "/content/drive/MyDrive/llm_from_scratch/src/"
filename = url.split('/')[-1]
full_path = os.path.join(download_path, filename)
print(full_path)
urllib.request.urlretrieve(url, full_path)

/content/drive/MyDrive/llm_from_scratch/src/gpt_download.py


('/content/drive/MyDrive/llm_from_scratch/src/gpt_download.py',
 <http.client.HTTPMessage at 0x7f1e22f4c200>)

In [ ]:
import torch
import torch.nn as nn

In [ ]:
from gpt_download import download_and_load_gpt2

In [ ]:
settings, params = download_and_load_gpt2(model_size = '774M', models_dir = 'gpt2')

File already exists and is up-to-date: gpt2/774M/checkpoint
File already exists and is up-to-date: gpt2/774M/encoder.json
File already exists and is up-to-date: gpt2/774M/hparams.json
File already exists and is up-to-date: gpt2/774M/model.ckpt.data-00000-of-00001
File already exists and is up-to-date: gpt2/774M/model.ckpt.index
File already exists and is up-to-date: gpt2/774M/model.ckpt.meta
File already exists and is up-to-date: gpt2/774M/vocab.bpe


In [ ]:
print(settings)
print(params.keys())

{'n_vocab': 50257, 'n_ctx': 1024, 'n_embd': 1280, 'n_head': 20, 'n_layer': 36}
dict_keys(['blocks', 'b', 'g', 'wpe', 'wte'])


In [ ]:
print(params['wte'].shape)

(50257, 1280)


In [ ]:
CONFIG = {
"vocab_size": 50257,
"context_length": 1024,
"emb_dim": 1280,
"n_heads": 20,
"n_layers": 36,
"drop_rate": 0.1,
"qkv_bias": True
}

In [ ]:
from gpt_model import GPTModel

In [ ]:
gpt = GPTModel(CONFIG)

In [ ]:
# code to load pretrained weights into out own model:
gpt.eval()
# first function to check tensor shape matches:
def assign(left, right):
    if left.shape != right.shape:
        raise ValueError(f"Shape mismatch. Left: {left.shape}, Right: {right.shape}")
    return torch.nn.Parameter(torch.tensor(right))

In [ ]:
import numpy as np
def load_weights_into_gpt(gpt, params):
    gpt.pos_emb.weight = assign(gpt.pos_emb.weight, params['wpe'])
    gpt.tok_emb.weight = assign(gpt.tok_emb.weight, params['wte'])
    for b in range(len(params["blocks"])):
        q_w, k_w, v_w = np.split(
        (params["blocks"][b]["attn"]["c_attn"])["w"], 3, axis=-1)
        gpt.trf_blocks[b].att.W_query.weight = assign(
            gpt.trf_blocks[b].att.W_query.weight, q_w.T)
        gpt.trf_blocks[b].att.W_key.weight = assign(
            gpt.trf_blocks[b].att.W_key.weight, k_w.T)
        gpt.trf_blocks[b].att.W_value.weight = assign(
            gpt.trf_blocks[b].att.W_value.weight, v_w.T)
        q_b, k_b, v_b = np.split(
            (params["blocks"][b]["attn"]["c_attn"])["b"], 3, axis=-1)
        gpt.trf_blocks[b].att.W_query.bias = assign(
            gpt.trf_blocks[b].att.W_query.bias, q_b)
        gpt.trf_blocks[b].att.W_key.bias = assign(
            gpt.trf_blocks[b].att.W_key.bias, k_b)
        gpt.trf_blocks[b].att.W_value.bias = assign(
            gpt.trf_blocks[b].att.W_value.bias, v_b)
        gpt.trf_blocks[b].att.out_proj.weight = assign(
            gpt.trf_blocks[b].att.out_proj.weight,
            params["blocks"][b]["attn"]["c_proj"]["w"].T)
        gpt.trf_blocks[b].att.out_proj.bias = assign(
            gpt.trf_blocks[b].att.out_proj.bias,
            params["blocks"][b]["attn"]["c_proj"]["b"])
        gpt.trf_blocks[b].ff.layers[0].weight = assign(
            gpt.trf_blocks[b].ff.layers[0].weight,
            params["blocks"][b]["mlp"]["c_fc"]["w"].T)
        gpt.trf_blocks[b].ff.layers[0].bias = assign(
            gpt.trf_blocks[b].ff.layers[0].bias,
            params["blocks"][b]["mlp"]["c_fc"]["b"])
        gpt.trf_blocks[b].ff.layers[2].weight = assign(
            gpt.trf_blocks[b].ff.layers[2].weight,
            params["blocks"][b]["mlp"]["c_proj"]["w"].T)
        gpt.trf_blocks[b].ff.layers[2].bias = assign(
            gpt.trf_blocks[b].ff.layers[2].bias,
            params["blocks"][b]["mlp"]["c_proj"]["b"])
        gpt.trf_blocks[b].norm1.scale = assign(
            gpt.trf_blocks[b].norm1.scale,
            params["blocks"][b]["ln_1"]["g"])
        gpt.trf_blocks[b].norm1.shift = assign(
            gpt.trf_blocks[b].norm1.shift,
            params["blocks"][b]["ln_1"]["b"])
        gpt.trf_blocks[b].norm2.scale = assign(
            gpt.trf_blocks[b].norm2.scale,
            params["blocks"][b]["ln_2"]["g"])
        gpt.trf_blocks[b].norm2.shift = assign(
            gpt.trf_blocks[b].norm2.shift,
            params["blocks"][b]["ln_2"]["b"])

    gpt.final_norm.scale = assign(gpt.final_norm.scale, params["g"])
    gpt.final_norm.shift = assign(gpt.final_norm.shift, params["b"])
    gpt.out_head.weight = assign(gpt.out_head.weight, params["wte"])

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
load_weights_into_gpt(gpt, params)
gpt.to(device)

GPTModel(
  (tok_emb): Embedding(50257, 1280)
  (pos_emb): Embedding(1024, 1280)
  (drop_emb): Dropout(p=0.1, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (dropout): Dropout(p=0.1, inplace=False)
        (W_query): Linear(in_features=1280, out_features=1280, bias=True)
        (W_key): Linear(in_features=1280, out_features=1280, bias=True)
        (W_value): Linear(in_features=1280, out_features=1280, bias=True)
        (out_proj): Linear(in_features=1280, out_features=1280, bias=True)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=1280, out_features=5120, bias=True)
          (1): GELU()
          (2): Linear(in_features=5120, out_features=1280, bias=True)
        )
      )
      (norm1): LayerNormalization()
      (norm2): LayerNormalization()
      (drop_shortcut): Dropout(p=0.1, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (

In [ ]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")
from functions_to_generate_text import generate, token_ids_to_text, text_to_token_ids

In [ ]:
torch.manual_seed(123)
token_ids = generate(
model=gpt,
idx=text_to_token_ids("Drug addiction is a", tokenizer),
max_new_tokens=500,
context_size=CONFIG["context_length"],
top_k=100,
temperature=0.6
)
print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

Output text:
 Drug addiction is a disease, the result of a chemical imbalance in the brain. It's a disease of the brain. It's not a disease of the body. Unfortunately, a lot of people have been trying to blame the body for this. And that is not the way this disease works. It works in the brain, in the hypothalamus. It works in the brain's reward center. It works in the brain's reward center. And it works in the brain's reward center. And it's a very old disease. It's been around for thousands of years. It's not new. It's been around for thousands of years. And it's going to be around for thousands more.

So what we have to do is stop trying to blame the brain for this. That's not how this disease works. It works in the brain. And we've got to stop trying to blame the brain for this. And we've got to stop trying to blame the brain for this. And we've got to start listening to the brain. And we've got to start listening to the brain. And we've got to start listening to the brain. And we'